In [1]:
bibliography_file = '../../../data/biblio-v5-table.ods'

In [2]:
import polars as pl

In [3]:
df_raw = pl.read_ods(bibliography_file)

In [4]:
# Process the 'pages' column to split it into 'start' and 'end' pages by "--"

df = df_raw.select([
    "bibkey",
    "title",
    "author",
    "date",
    "journal",
    "volume",
    "number",
    "pages"
]).with_columns([
    pl.col('pages')
    .str.strip_chars()
    .str.split('--')
    .alias('pages_split')
]).with_columns([
    pl.col('pages_split').list.get(0, null_on_oob=True).alias('start'),
    pl.col('pages_split').list.get(1, null_on_oob=True).alias('end')
]).drop('pages_split').with_columns([
    pl.col('start').str.replace_all(r'[^0-9]', '').cast(pl.Int32, strict=False).alias('start_int'),
])

In [5]:
sorted = df.sort('start_int', nulls_last=True).sort("number", descending=True).sort('volume', descending=True).sort('date', descending=True)

In [6]:

nous = sorted.filter(pl.col('journal') == 'No{\\^u}s')

In [7]:
#journal_items.sort('date').sort("volume").sort("number").sort('start')

apq = sorted.filter(pl.col('journal') == 'American Philosophical Quarterly')

In [8]:
apq

bibkey,title,author,date,journal,volume,number,pages,start,end,start_int
str,str,str,i64,str,i64,str,str,str,str,i32
"""duncan_m:2021""","""Animalism is Either False or U…","""Duncan, Matt""",2021,"""American Philosophical Quarter…",58,"""2""","""187--200""","""187""","""200""",187
"""thornton_ak:2019""","""Disembodied Animals""","""Thornton, Allison Krile""",2019,"""American Philosophical Quarter…",56,"""2""","""203--217""","""203""","""217""",203
"""wedgwood_r:2019""","""Moral Disagreement and Inexcus…","""Wedgwood, Ralph""",2019,"""American Philosophical Quarter…",56,"""1""","""97--108""","""97""","""108""",97
"""cowling_s-cray:2017""","""How to Be Omnipresent""","""Cowling, Sam and Cray, Wesley …",2017,"""American Philosophical Quarter…",54,"""3""","""223--234""","""223""","""234""",223
"""bourget:2017""","""Representationalism and Sensor…","""Bourget, David""",2017,"""American Philosophical Quarter…",54,"""3""","""251--268""","""251""","""268""",251
…,…,…,…,…,…,…,…,…,…,…
"""chisholm:1964""","""The Ethics of Requirement""","""Chisholm, Roderick M.""",1964,"""American Philosophical Quarter…",1,null,"""147--153""","""147""","""153""",147
"""thalberg:1964a""","""Emotion and Thought""","""Thalberg, Irving""",1964,"""American Philosophical Quarter…",1,null,null,null,null,null
"""spiegelberg:1964""","""Toward a Phenomenology of Expe…","""Spiegelberg, Herbert""",1964,"""American Philosophical Quarter…",1,"""4""","""325--332""","""325""","""332""",325


In [9]:
import sys

sys.path.append("../..")

In [10]:
from ref_pipe.models import HTMLIssue, HTMLVolume, HTMLYear, THTMLCollapsible  # type: ignore


def build_collapsible(df: pl.DataFrame) -> THTMLCollapsible:
    years = []
    for year_val, year_group in df.group_by("date", maintain_order=True):
        year_str = " ".join(map(str, year_val))

        volumes = year_group.select("volume").unique().drop_nulls().to_series().to_list()
        has_multiple_volumes = len(volumes) > 1

        bibs_without_volume = year_group.filter(pl.col("volume").is_null())["bibkey"].to_list()

        if not has_multiple_volumes:
            # Single or no volume: flatten to HTMLYear
            year_bibkeys = year_group["bibkey"].to_list()
            name = year_str
            if len(volumes) == 1:
                name += f" :: vol. {volumes[0]}"
            years.append(HTMLYear(name=name, contents=tuple(year_bibkeys)))
        else:
            # Multiple volumes
            year_contents = [*bibs_without_volume]
            volume_names_g = (
                f"{v}" for v in year_group["volume"].unique().drop_nulls().to_list()
            )
            volume_names = ", ".join(volume_names_g)
            year_name = f"{year_str} :: vol. {volume_names}"

            for volume_val, vol_group in year_group.group_by("volume", maintain_order=True):
                volume_str = " ".join(map(str, volume_val))

                volume_bibkeys_without_issue = vol_group.filter(pl.col("number").is_null().or_(pl.col("number") == ""))["bibkey"].to_list()
                volume_contents = [*volume_bibkeys_without_issue]

                issues = vol_group.select("number").unique().drop_nulls().to_series().to_list()
                for issue_val in issues:
                    issue_str = " ".join(map(str, issue_val))
                    issue_name = f"issue {issue_str}"

                    issue_bibkeys = vol_group.filter(pl.col("number") == issue_val)["bibkey"].to_list()
                    volume_contents.append(
                        HTMLIssue(name=str(issue_name), contents=tuple(issue_bibkeys))
                    )

                year_contents.append(HTMLVolume(name=volume_str, contents=tuple(volume_contents)))

            years.append(HTMLYear(name=year_name, contents=tuple(year_contents)))

    return tuple(years)


In [11]:
struct = build_collapsible(apq)

In [12]:
from pprint import pprint
pprint(struct)

(HTMLYear(name='2021 :: vol. 58', contents=('duncan_m:2021',)),
 HTMLYear(name='2019 :: vol. 56',
          contents=('thornton_ak:2019', 'wedgwood_r:2019')),
 HTMLYear(name='2017 :: vol. 54',
          contents=('cowling_s-cray:2017',
                    'bourget:2017',
                    'brunero:2017',
                    'gardiner_g:2017',
                    'miller_k:2017',
                    'shiller:2017',
                    'smith_njj:2017b',
                    'fuqua:2017',
                    'dunn_j-ahlstromvij:2017',
                    'hales_sd:2017',
                    'rescher:2017',
                    'bozickovic:2017',
                    'menges_al:2017a',
                    'turri-buckwalter:2017',
                    'forrest_p:2017b',
                    'fresco-etal:2017',
                    'cornell_dm:2017',
                    'sinhababu:2017a')),
 HTMLYear(name='2016 :: vol. 53',
          contents=('wyatt_j-lynch:2016',
                    'kind:201

In [13]:
struct_a = [x for x in struct if x.name == "1982"]

In [14]:
for x in struct:
    if isinstance(x, HTMLYear):
        print(f"  Year {x.name}")
        for y in x.contents:
            if isinstance(y, HTMLVolume):
                for z in y.contents:
                    print(f"      Volume {y.name}")
                    if isinstance(z, HTMLIssue):
                        for a in z.contents:
                            print(f"          Issue {z.name}")
            else:
                print("      ", y)
    else:
        print("  ", x)

  Year 2021 :: vol. 58
       duncan_m:2021
  Year 2019 :: vol. 56
       thornton_ak:2019
       wedgwood_r:2019
  Year 2017 :: vol. 54
       cowling_s-cray:2017
       bourget:2017
       brunero:2017
       gardiner_g:2017
       miller_k:2017
       shiller:2017
       smith_njj:2017b
       fuqua:2017
       dunn_j-ahlstromvij:2017
       hales_sd:2017
       rescher:2017
       bozickovic:2017
       menges_al:2017a
       turri-buckwalter:2017
       forrest_p:2017b
       fresco-etal:2017
       cornell_dm:2017
       sinhababu:2017a
  Year 2016 :: vol. 53
       wyatt_j-lynch:2016
       kind:2016d
       vallier:2016
       roche_m:2016
       thompson_n:2016a
       simion-kelp:2016
       dorsey_je:2016
       fazekas_k:2016
       carter_ja-palermos:2016
       mckenna_r:2016
       lemos_nm:2016
       capes_ja:2016a
       manela:2016
       masto:2016
       mcconnell_d:2016
       farkas_k:2016
       frances_b:2016b
       kersten_l-wilson:2016
       neta:2016a
    

In [15]:
from ref_pipe.models import TBibkey, Bibliography
from aletk.utils import get_logger

lgr = get_logger(__name__)

year_format = """
<details class="collapsible-level-1">
    <summary>{name}</summary>
    <hr>

    {contents}

</details>
"""

volume_format = """
<details class="collapsible-level-2">
    <summary>{name}</summary>
    <hr>

    {contents}
</details>
"""

issue_format = """
<details class="collapsible-level-3">
    <summary>{name}</summary>
    <hr>

    {contents}
</details>
"""

def _get_bibkey_html(bibkey: TBibkey, bibliography: Bibliography) -> str:
    index = bibliography.bibkey_index_dict.get(bibkey, None)
    if index is None:
        lgr.warning(f"Bibkey {bibkey} not found in bibliography.")
        return ""
    if not isinstance(index, int) or index < 0:
        lgr.warning(f"Bibkey index {index} is not valid.")
        return "" 
    
    try:
        html = bibliography.content[index]
        assert isinstance(html, str)

    except Exception as e:
        lgr.warning(f"Error getting HTML for bibkey '{bibkey}': {e}")
        return ""

    return html

In [16]:
from typing import Callable
from ref_pipe.models import Bibliography, TBibkey, HTMLIssue, HTMLVolume, THTMLCollapsible

type TGetBibkeyHTML = Callable[
    [TBibkey], str,
]

def generate_struct_html(struct: THTMLCollapsible, get_bibkey_html: TGetBibkeyHTML) -> str:
    main_html = f""

    for year in struct:

        year_html_contents = f""
        for year_content in year.contents:

            if isinstance(year_content, str):
                year_html_contents += get_bibkey_html(year_content)
                year_html_contents += "\n"

            elif isinstance(year_content, HTMLVolume):

                vol_html_contents = f""
                for vol_content in year_content.contents:

                    if isinstance(vol_content, str):
                        vol_html_contents += get_bibkey_html(vol_content)
                        vol_html_contents += "\n"

                    elif isinstance(vol_content, HTMLIssue):
                        issue_html_contents = f""

                        for issue_content in vol_content.contents:
                            issue_html_contents += get_bibkey_html(issue_content)
                            issue_html_contents += "\n"
                        
                        issue_html = issue_format.format(
                            name=vol_content.name,
                            contents=issue_html_contents
                        )
                        vol_html_contents += issue_html
                        vol_html_contents += "\n"
                    else:
                        raise ValueError(f"Unknown content type for volume content: {type(vol_content)}")

                volume_html = volume_format.format(
                    name=year_content.name,
                    contents=vol_html_contents
                )
                year_html_contents += volume_html
                year_html_contents += "\n"

            else:
                raise ValueError(f"Unknown content type for year content: {type(year_content)}")


        year_html = year_format.format(
            name=year.name,
            contents=year_html_contents
        )
        main_html += year_html
        main_html += "\n"

    return main_html



        

In [17]:
from src.ref_pipe.filesystem_io import load_bibliography
from src.sdk.ResultMonad import runwrap


local_bibliography_filepath = "/home/alebg/philosophie-ch/Dropbox/philosophie-ch/dltc-biblio/biblio-v5.bib"

bibliography = runwrap(load_bibliography(local_bibliography_filepath))
get_bibkey_html: TGetBibkeyHTML = lambda bibkey: _get_bibkey_html(bibkey, bibliography)


html = generate_struct_html(struct, get_bibkey_html)

2025-04-27 12:47:48 - Filesystem I/O - INFO - load_bibliography_bibkeys
	Reading bibliography file '/home/alebg/philosophie-ch/Dropbox/philosophie-ch/dltc-biblio/biblio-v5.bib'...
2025-04-27 12:47:49 - Bibkey Utils - WARNING - Unexpected year suffix for 'davidson_d:1999z10': 'z10'
2025-04-27 12:47:49 - Bibkey Utils - WARNING - Unexpected year suffix for 'davidson_d:1999z17': 'z17'
2025-04-27 12:47:49 - Bibkey Utils - WARNING - Unexpected year suffix for 'tymieniecka:1988z7': 'z7'
2025-04-27 12:47:49 - Bibkey Utils - WARNING - Unexpected year suffix for 'tymieniecka:2000z19': 'z19'
2025-04-27 12:47:49 - Bibkey Utils - WARNING - Unexpected year suffix for 'tymieniecka:2000z10': 'z10'
2025-04-27 12:47:49 - Bibkey Utils - WARNING - Unexpected year suffix for 'tymieniecka:2000z2': 'z2'
2025-04-27 12:47:49 - Bibkey Utils - WARNING - Unexpected year suffix for 'nasr_sh:2001z2': 'z2'
2025-04-27 12:47:49 - Bibkey Utils - WARNING - Unexpected year suffix for 'tymieniecka:2000z1': 'z1'
2025-04-27

In [19]:
with open("apq.html", "w") as f:
    f.write(html)
    f.write("\n")